# Importare le librerie

In [11]:
import easyocr
reader = easyocr.Reader(["it"])
import os
import pandas as pd
import cv2
import numpy as np
import csv

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


# Definire le funzioni

In [53]:
def prima_lettura(png):
    result = reader.readtext(
        png,
        detail=1,
        paragraph=False,
        mag_ratio=2
    )
    return result

In [83]:
def zona(png):
    # Cercare le righe DESCRIZIONE e SUBTOTALE
    y_descrizione = None
    y_subtotale = None

    for bbox, txt, conf in result:
        testo = txt.upper()

        if "DESCRIZIONE" in testo:
            y_descrizione = min(p[1] for p in bbox)

        if "SUBTOTALE" in testo or "SUFFQTALOMPLESSIVO" in testo: #Ho aggiunto una lettura alternativa
            y_subtotale = min(p[1] for p in bbox)

    # Tagliare solo la zona articoli
    img = cv2.imread(png)
    altezza, larghezza = img.shape[:2]

    zona_articoli = img[
        y_descrizione:y_subtotale,
        0:larghezza
    ]

    # Ingrandire
    def ingrandisci(img):
        return cv2.resize(
            img,
            None,
            fx=4,
            fy=4,
            interpolation=cv2.INTER_CUBIC
        )

    zona_articoli = ingrandisci(zona_articoli)
    return zona_articoli

In [55]:
def seconda_lettura(zona):
    ocr_ingrandito = reader.readtext(
        zona,
        detail=1,
        paragraph=False
        )
    return ocr_ingrandito

In [56]:
tolleranza_y = 50

In [57]:
def righe_da_mantenere(ocr_ingrandito):
    df_ocr_ingrandito = pd.DataFrame([
    {"Descrizione" : text,
     "Confidenza" : round(conf, 2),
     "x1" : round(bbox[0][0]), "y1" : round(bbox[0][1]),
     "x2" : round(bbox[1][0]), "y2" : round(bbox[1][1]),
     "x3" : round(bbox[2][0]), "y3" : round(bbox[2][1]),
     "x4" : round(bbox[3][0]), "y4" : round(bbox[3][1])}
     for bbox, text, conf in ocr_ingrandito]
    )

    df_ocr_ingrandito["Descrizione"] = df_ocr_ingrandito["Descrizione"].str.replace(" ", "")

    indici_righe = [0]

    for x in range(1, len(df_ocr_ingrandito["Descrizione"])):
        attuale = df_ocr_ingrandito["y1"][x],
        precedente = df_ocr_ingrandito["y1"][x - 1]
        if (precedente - tolleranza_y) <=  attuale <= (precedente + tolleranza_y):
            pass
        else:
            indici_righe.append(x)

    riga_precedente = indici_righe[0]
    righe = []
    range_righe = []

    for x in indici_righe[1:]:
        indice_riga_completa = range(riga_precedente, x)
        # print(df_ocr_ingrandito["Descrizione"][indice_riga_completa])
        righe.append(" ".join(df_ocr_ingrandito["Descrizione"][indice_riga_completa].astype(str)))
        range_righe.append(indice_riga_completa)
        riga_precedente = x

    righe.append(" ".join(df_ocr_ingrandito["Descrizione"][range(indici_righe[-1], len(df_ocr_ingrandito["Descrizione"]))].astype(str)))
    range_righe.append(range(riga_precedente, len(df_ocr_ingrandito["Descrizione"])))

    # print("Ci sono", len(indici_righe), "indici e", len(righe), "righe")

    # Creo il DF impostando come indice la colonna indici_righe
    df_righe = pd.DataFrame({"Indici" : indici_righe,
                            "Testo" : righe
                            })
    df_righe = df_righe.set_index("Indici")#.reset_index(inplace=True)

    # Ottengo gli indici delle righe da eliminare per il Cad e Pz
    filtro_Cad = df_righe["Testo"].str.contains("Cad") & \
            df_righe["Testo"].str.contains("Pz|PZ")
    righe_eliminare_Cad = df_righe[filtro_Cad].index.tolist()
    # print("Righe da eliminare per Cad Pz", righe_eliminare_Cad)

    # Ottengo gli indici delle righe da eliminare per kg o EUR/kg
    filtro_kg = df_righe["Testo"].str.contains("kg|EUR/kg")
    righe_eliminare_kg = df_righe[filtro_kg].index.tolist()
    # print("Righe da eliminare per kg o EUR/kg", righe_eliminare_kg)

    # Ottengo gli indici delle righe da eliminare per Offerta Lidl Plus
    filtro_offerta = df_righe["Testo"].str.contains("Offerta Lidl Plus")
    righe_eliminare_offerta = df_righe[filtro_offerta].index.tolist()
    # print("Righe da eliminare per Offerta Lidl Plus", righe_eliminare_offerta)

    # Ottengo gli indici delle righe da eliminare per Valore sconti
    filtro_sconti = df_righe["Testo"].str.contains("VALORE SCONTI")
    righe_eliminare_sconti = df_righe[filtro_sconti].index.tolist()
    # print("Righe da eliminare per Valore sconti", righe_eliminare_sconti)

    righe_eliminare_complessivo = righe_eliminare_Cad.copy()
    righe_eliminare_complessivo.extend(righe_eliminare_kg)
    righe_eliminare_complessivo.extend(righe_eliminare_offerta)
    righe_eliminare_complessivo.extend(righe_eliminare_sconti)
    indici_righe_pulite = [x for x in indici_righe if x not in righe_eliminare_complessivo]
    # print(righe_eliminare_complessivo)
    # print(indici_righe)
    # print(indici_righe_pulite)

    range_righe_tenere = []
    range_righe_scartare = []

    for x in range(len(range_righe)):
        if range_righe[x][0] in indici_righe_pulite:
                Esito = "True"
                range_righe_tenere.append(range_righe[x])
        else:
            Esito = "False"
            range_righe_scartare.append(range_righe[x])
            # print(range_righe[x][0], indici_righe[x], Esito)

    righe_tenere = [x for r in range_righe_tenere for x in r]
    df_ocr_ingrandito_clean = df_ocr_ingrandito.loc[righe_tenere]
    
    return df_ocr_ingrandito_clean

In [58]:
def prezzi(df_ocr_ingrandito_clean):
    df_prezzi = df_ocr_ingrandito_clean[3:].loc[df_ocr_ingrandito_clean["x1"] > 3000].sort_values("y1")
    df_prezzi["riga"] = (df_prezzi["y1"].diff().abs() >= tolleranza_y).cumsum()
    df_prezzi = df_prezzi.sort_values(["riga", "x1"])
    df_prezzi["Descrizione"] = (df_prezzi.groupby("riga")["Descrizione"].transform(" ".join))
    df_prezzi = df_prezzi.drop_duplicates("riga").drop(columns="riga")
    df_prezzi["Prezzo proposto"] = df_prezzi["Descrizione"].loc[df_prezzi["Descrizione"].str.contains(" ")]
    df_prezzi["Prezzo proposto"] = df_prezzi["Prezzo proposto"].str.replace({",": "",
                                                                            "=": "",
                                                                            "E": "",})
    df_prezzi["Prezzo proposto"] = df_prezzi["Prezzo proposto"].str.replace(" ", ".")
    df_prezzi.loc[df_prezzi["Descrizione"].str.contains(" "), "Descrizione"] = np.nan
    df_prezzi["Descrizione"] = df_prezzi["Descrizione"].str.replace(",", ".").astype(float)
    return df_prezzi

In [59]:
def iva(df_ocr_ingrandito_clean):
    df_iva = df_ocr_ingrandito_clean[3:].loc[(df_ocr_ingrandito_clean["x1"] > 2500) & (df_ocr_ingrandito_clean["x1"] <= 3000)].sort_values("y1")
    df_iva["Descrizione"] = df_iva["Descrizione"].str.replace({"22%" : "22",
                                                            "229" : "22",
                                                            "10%" : "10",
                                                            "0%" : "10",
                                                            "109" : "10",
                                                            "4%" : "4"}).astype(int)
    return df_iva

In [60]:
def articoli(df_ocr_ingrandito_clean):
    df_articoli = df_ocr_ingrandito_clean[3:].loc[df_ocr_ingrandito_clean["x1"] <= 2500].sort_values("y1")
    df_articoli["riga"] = (df_articoli["y1"].diff().abs() >= tolleranza_y).cumsum()
    df_articoli = df_articoli.sort_values(["riga", "x1"]) #senza questo le parole all'interno della riga erano in ordine sparso
    df_articoli["Descrizione"] = (df_articoli.groupby("riga")["Descrizione"].transform(" ".join))
    df_articoli = df_articoli.drop_duplicates("riga").drop(columns="riga")
    return df_articoli

In [61]:
def df_complessivo(df_articoli, df_iva, df_prezzi):
    df_finale = pd.concat([df_articoli["Descrizione"].reset_index(drop=True),
                        df_iva["Descrizione"].reset_index(drop=True),
                        df_prezzi["Descrizione"].reset_index(drop=True),
                        df_prezzi["Prezzo proposto"].reset_index(drop=True)],
                        axis = 1)
    df_finale.columns = ["Articolo", "IVA", "Prezzo", "Prezzo proposto"]
    return df_finale

In [62]:
def zona_info_scontrino(result):
    # Cercare le righe DESCRIZIONE e DOCUMENTO
    y_importo = None
    y_documento = None

    for bbox, txt, conf in result:
        testo = txt.upper()

        if "PAGATO" in testo: #"IMPORTO"
            y_importo = min(p[1] for p in bbox)

        if "DOCUMENTO" in testo:
            y_documento = min(p[1] for p in bbox)

    if y_importo is not None and y_documento is not None:

        risultati_filtrati = [
            (bbox, txt, conf)
            for bbox, txt, conf in result
            if y_importo <= min(p[1] for p in bbox) < y_documento
        ]
    return risultati_filtrati

In [63]:
def estrazione_data_ora(risultati_filtrati):
    df_risultati_filtrati = pd.DataFrame([
        {"Descrizione" : text,
        "Confidenza" : round(conf, 2),
        "x1" : round(bbox[0][0]), "y1" : round(bbox[0][1]),
        "x2" : round(bbox[1][0]), "y2" : round(bbox[1][1]),
        "x3" : round(bbox[2][0]), "y3" : round(bbox[2][1]),
        "x4" : round(bbox[3][0]), "y4" : round(bbox[3][1])}
        for bbox, text, conf in risultati_filtrati]
    )
    y_importo = df_risultati_filtrati["y1"][0]
    y_data_ora = y_importo + 100
    tolleranza_data_ora = 20

    df_risultati_filtrati = df_risultati_filtrati.loc[(df_risultati_filtrati["y1"] >= (y_data_ora - tolleranza_data_ora)) &
                            (df_risultati_filtrati["y1"] <= (y_data_ora + tolleranza_data_ora))].reset_index(drop=True)

    # La data viene spesso separata in più letture, ma a volte succede anche con l'ora. Quindi devo fare che la data è dal primo elemento fino all'anno
    indice_anno = df_risultati_filtrati.loc[df_risultati_filtrati["Descrizione"].str.contains("2026")].index[0] +1
    df_risultati_filtrati_data = df_risultati_filtrati.iloc[:indice_anno]
    df_risultati_filtrati_data = df_risultati_filtrati_data.sort_values("x1")
    Data = "-".join(df_risultati_filtrati_data["Descrizione"])

    df_risultati_filtrati_ora = df_risultati_filtrati.iloc[indice_anno:]
    df_risultati_filtrati_ora = df_risultati_filtrati_ora.sort_values("x1")
    Ora = ":".join(df_risultati_filtrati_ora["Descrizione"]).replace(" ", "")
    return Data, Ora

In [64]:
def aggiornamento_elenco(png, indice):
    with open("Elenco scontrini letti.csv", "a") as f:
        f.write(f"{png},")
        f.write(f"{indice},")
        f.write(f"{Data} {Ora}")
        f.write("\n")

# Processo principale

Processo:
1. Identificare gli scontrini di cui ho immagini
2. Controllare quali ho già letto nel file elenco scontrini
3. Processare gli scontrini nuovi
4. Scrivere nel file elenco scontrini che ho processato anche gli scontrini nuovi

In [84]:
cwd = os.getcwd()
raw_data = os.path.join(cwd, "raw_data")
png_files = [f for f in os.listdir(raw_data) if f.endswith(".png")]
processed_png = pd.read_csv("Elenco scontrini letti.csv")["Scontrino"].tolist()
png_to_process = [f for f in png_files if f not in processed_png]

if len(png_to_process) == 0:
    print("Non ci sono file da processare")
else:
    print("Devo processare i file", png_to_process)

ultimo_indice = pd.read_csv("Elenco scontrini letti.csv")["Indice"].tolist()[-1]
indici_to_process = list(range(ultimo_indice +1, ultimo_indice +1 +len(png_to_process)))

Devo processare i file ['Scontrino_prova10.png']


Qui eseguo tutto il codice, con alla fine l'aggiunta della colonna indice nel ciclo for

In [85]:
for png,indice in zip(png_to_process, indici_to_process):
    print("Inizio a processare il file", png)
    png_da_leggere = os.path.join(cwd, "raw_data", png)
    result = prima_lettura(png_da_leggere)
    zona_articoli = zona(png_da_leggere)
    ocr_ingrandito = seconda_lettura(zona_articoli)
    df_ocr_ingrandito_clean = righe_da_mantenere(ocr_ingrandito)
    df_prezzi = prezzi(df_ocr_ingrandito_clean)
    df_iva = iva(df_ocr_ingrandito_clean)
    df_articoli = articoli(df_ocr_ingrandito_clean)
    df_finale = df_complessivo(df_articoli, df_iva, df_prezzi)
    df_finale["Indice scontrino"] = indice
    df_finale.to_csv("Database.csv", mode="a", header=False, index=False)
    risultati_filtrati = zona_info_scontrino(result)
    Data, Ora = estrazione_data_ora(risultati_filtrati)
    aggiornamento_elenco(png, indice)
    print("Completato", indici_to_process.index(indice) +1, "di", len(indici_to_process))

Inizio a processare il file Scontrino_prova10.png


c:\Users\Computer\miniconda3\envs\OCR\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Completato 1 di 1


# Correzioni manuali

## Creazione

In [ ]:
# database = pd.read_csv("Database.csv")
# database.loc[database["Prezzo"].isna()].to_csv("Corretti manualmente.csv", index=False)

,Articolo,IVA,Prezzo,Prezzo proposto,Indice scontrino


## Aggiornamento

In [86]:
database = pd.read_csv("Database.csv")
database = database.loc[database["Prezzo"].isna()]

# Aggiungo un nuovo articolo fittizio
# database.loc[len(database)] = ["Nuovo articolo", 4, 0.50, np.nan, 10]

In [87]:
articoli_corretti = pd.read_csv("Corretti manualmente.csv")
articoli_da_aggiungere = database.merge(articoli_corretti, how="outer", indicator=True)
articoli_da_aggiungere = articoli_da_aggiungere[articoli_da_aggiungere["_merge"] == "left_only"].drop(columns="_merge")

In [88]:
with open("Corretti manualmente.csv", "a", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    for riga in articoli_da_aggiungere.itertuples(index=False, name=None):
        writer.writerow(riga)